In [ ]:
import json
import os
from pathlib import Path

def extract_cve_cwe_mappings(cve_dir):
    mappings = {}
    for year_dir in Path(cve_dir).iterdir():
        # print(year_dir)
        if year_dir.is_dir():
            # print(year_dir)
            for cve_file in year_dir.glob('*.json'):
                print(cve_file)
                with open(cve_file) as f:
                    try:
                        cve_data = json.load(f)
                        print(cve_data)
                        cve_id = cve_data.get('cveMetadata', {}).get('cveId')
                        problem_types = cve_data.get('containers', {}).get('cna', {}).get('problemTypes', [])
                        cwes = []
                        for pt in problem_types:
                            for desc in pt.get('descriptions', []):
                                if desc.get('type') == 'CWE':
                                    cwes.append(desc.get('description'))
                        if cve_id and cwes:
                            mappings[cve_id] = cwes
                    except json.JSONDecodeError:
                        continue
                break
    return mappings

# Usage
cve_cwe_mappings = extract_cve_cwe_mappings('cvelistV5/cves')

print(cve_cwe_mappings)

In [ ]:
import json
from pathlib import Path

def extract_cve_cwe_mappings(cve_dir):
    mappings = {}
    # Iterate through year directories (1999, 2000, etc.)
    for year_dir in Path(cve_dir).iterdir():
        if year_dir.is_dir() and year_dir.name.isdigit():
            # print(f"Processing year: {year_dir.name}")
            
            # Now iterate through the xxx subdirectories
            for subdir in year_dir.iterdir():
                if subdir.is_dir():
                    # Now look for JSON files in these subdirectories
                    for cve_file in subdir.glob('*.json'):
                        # print(f"Found CVE file: {cve_file}")
                        try:
                            with open(cve_file) as f:
                                cve_data = json.load(f)
                                cve_id = cve_data.get('cveMetadata', {}).get('cveId')
                                problem_types = cve_data.get('containers', {}).get('cna', {}).get('problemTypes', [])
                                cwes = []
                                for pt in problem_types:
                                    for desc in pt.get('descriptions', []):
                                        if desc.get('type') == 'CWE':
                                            cwes.append(desc.get('description'))
                                if cve_id and cwes:
                                    mappings[cve_id] = cwes
                        except (json.JSONDecodeError, UnicodeDecodeError) as e:
                            print(f"Error processing {cve_file}: {str(e)}")
                            continue
    return mappings

# Usage - point this to the 'cves' directory in the cloned repo
cve_cwe_mappings = extract_cve_cwe_mappings('cvelistV5/cves')

# Print some results
for i, (cve, cwes) in enumerate(cve_cwe_mappings.items()):
    print(f"{cve}: {cwes}")
    if i > 10:  # Just print the first 10 for demonstration
        break


In [ ]:
# type(cve_cwe_mappings)
# import csv
# # Write the list of dictionaries to a CSV file
# with open('./CVE_CWE_mapping.csv', mode='w', newline='') as file:
#     writer = csv.writer(file)
    
#     # Write the header
#     writer.writerow(['Key', 'Value'])
    
#     # Write each key-value pair as a new row
#     for key, value in cve_cwe_mappings.items():
#         writer.writerow([key, value])

In [ ]:
# test

In [ ]:
import pandas as pd
import requests

# Load your CSV
df = pd.read_csv('./CVE_CWE_mapping.csv')

# Example using NVD API
# get the exploits
def get_cve_details(cve_id):
    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
    response = requests.get(url)
    return response.json()

# Example using AlienVault OTX
def get_iocs(cve_id):
    url = f"https://otx.alienvault.com/api/v1/indicators/cve/{cve_id}"
    response = requests.get(url)
    return response.json()


In [ ]:
get_cve_details('CVE-2017-3793')

In [ ]:
get_iocs('CVE-2017-3793')

In [ ]:
import pandas as pd
import requests
import time
import json

# Load your CSV
df = pd.read_csv('./CVE_CWE_mapping.csv')

# API configuration
API_DELAY = 1  # seconds between API calls to avoid rate limiting

def get_iocs(cve_id):
    """Get IOCs from AlienVault OTX"""
    time.sleep(API_DELAY)  # Respect rate limits
    try:
        url = f"https://otx.alienvault.com/api/v1/indicators/cve/{cve_id}"
        response = requests.get(url)
        response.raise_for_status()
        
        # Extract relevant IOCs from response
        iocs = {
            'Pulse_info': response.json().get('pulse_info', []),
            'Exploits': response.json().get('exploits', [])
        }
        
        return response.json(), iocs
    except Exception as e:
        print(f"Error fetching IOCs for {cve_id}: {str(e)}")
        return None

def get_exploits(cve_id):
    """Get exploit information from ExploitDB and other sources"""
    time.sleep(API_DELAY)
    try:
        # Check ExploitDB
        exploitdb_url = f"https://www.exploit-db.com/search?cve={cve_id}"
        # Alternatively use the official API if you have access

        # Check alient_vault for known exploits
        alient_vault_url = f"https://otx.alienvault.com/api/v1/indicators/cve/{cve_id}"
        alient_vault_response = requests.get(alient_vault_url)
        alient_vault_response.raise_for_status()
        
        # Check NVD for known exploits _ i foudn that it gives reference to lots of website giving exploits but not the exploits directly
        nvd_url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
        nvd_response = requests.get(nvd_url)
        nvd_data = nvd_response.json()
        
        exploits = {
            'ExploitDB': exploitdb_url,  # Would be better with API
            'AlienVault_exploits': alient_vault_response.json().get('exploits', []),
            # 'NVD_References': [ref['url'] for ref in nvd_data['vulnerabilities'][0]['cve']['references']],
            'Published_Exploits': len(alient_vault_response.json().get('exploits', [])),
            'Manual checks at': [exploitdb_url, alient_vault_url]
        }
        return exploits
    except Exception as e:
        print(f"Error fetching exploits for {cve_id}: {str(e)}")
        return None

In [ ]:
thejson, theparsedresult = get_iocs('CVE-2017-5638')

# json_file = 'output.json'
# # Step 5: Write the dictionary to a JSON file
# with open(json_file, 'w') as file:
#     json.dump(thejson, file, indent=4)
        
# for key, value in thejson.items():
#     print(f"{key}: {value}")

print(theparsedresult)

In [ ]:
get_exploits('CVE-2017-5638')

In [11]:
import pandas as pd
import requests
import time
import json


def get_iocsANDexploits(cve_id):
    """Enhanced IOC extraction from multiple sources with better OTX parsing"""
    time.sleep(API_DELAY)
    try:
        # First try AlienVault OTX
        otx_url = f"https://otx.alienvault.com/api/v1/indicators/cve/{cve_id}"
        otx_response = requests.get(otx_url)
        otx_response.raise_for_status()
        otx_data = otx_response.json()
        
        iocs = {
            'IPs': set(),
            'Domains': set(),
            'URLs': set(),
            'Hashes': set(),
            'FileNames': set(),
            'Emails': set(),
            'Pulse_count': 0,
            'Pulse_samples': []
        }

        # Parse OTX pulses more thoroughly
        if 'pulse_info' in otx_data and 'pulses' in otx_data['pulse_info']:
            pulses = otx_data['pulse_info']['pulses']
            iocs['Pulse_count'] = len(pulses)
            
            # Sample first 3 pulse names for context
            iocs['Pulse_samples'] = [pulse.get('name', '') for pulse in pulses[:3]]
            
            # Try to get indicators from pulse details
            for pulse in pulses[:5]:  # Limit to first 5 pulses to avoid timeout
                try:
                    pulse_id = pulse['id']
                    pulse_detail_url = f"https://otx.alienvault.com/api/v1/pulses/{pulse_id}"
                    pulse_response = requests.get(pulse_detail_url, timeout=10)
                    pulse_data = pulse_response.json()
                    
                    if 'indicators' in pulse_data:
                        for indicator in pulse_data['indicators']:
                            indicator_type = indicator.get('type', '')
                            indicator_value = indicator.get('indicator', '')
                            
                            if indicator_type == 'IPv4':
                                iocs['IPs'].add(indicator_value)
                            elif indicator_type == 'domain':
                                iocs['Domains'].add(indicator_value)
                            elif indicator_type == 'URL':
                                iocs['URLs'].add(indicator_value)
                            elif indicator_type in ['FileHash-MD5', 'FileHash-SHA1', 'FileHash-SHA256']:
                                iocs['Hashes'].add(indicator_value)
                            elif indicator_type == 'filename':
                                iocs['FileNames'].add(indicator_value)
                            elif indicator_type == 'email':
                                iocs['Emails'].add(indicator_value)
                except Exception as pulse_error:
                    print(f"Error processing pulse {pulse_id}: {str(pulse_error)}")
                    continue

        # If still no IOCs, try additional sources
        if not any(iocs.values()):  # If all IOC lists are empty
            print(f"No IOCs found in OTX for {cve_id}, trying threatminer as additional sources")
            iocs.update(get_iocs_from_secondary_sources(cve_id))
        
        # Convert sets to lists
        for key in iocs:
            if isinstance(iocs[key], set):
                iocs[key] = list(iocs[key])
                
        return {
            'source': 'OTX alien vault / threatminer if alien vault empty',
            'Exploits': otx_response.json().get('exploits', []),
            'pulse_context': {
                'count': iocs.pop('Pulse_count'),
                'samples': iocs.pop('Pulse_samples')
            },
            'iocs': iocs
        }
        
    except Exception as e:
        print(f"Error fetching IOCs for {cve_id}: {str(e)}")
        return None

def get_iocs_from_secondary_sources(cve_id):
    """Fallback IOC extraction from other sources"""
    secondary_iocs = {
        'IPs': set(),
        'Domains': set(),
        'URLs': set(),
        'Hashes': set()
    }
    
    try:
        # Try ThreatMiner
        tm_url = f"https://api.threatminer.org/v2/vulnerability.php?q={cve_id}&api=True&rt=1"
        tm_response = requests.get(tm_url, timeout=10)
        if tm_response.status_code == 200:
            for result in tm_response.json().get('results', []):
                if result.get('type') == 'domain':
                    secondary_iocs['Domains'].add(result['value'])
                elif result.get('type') == 'ip':
                    secondary_iocs['IPs'].add(result['value'])
    
    except Exception as e:
        print(f"Error with ThreatMiner for {cve_id}: {str(e)}")
    
    return {k: list(v) for k, v in secondary_iocs.items() if v}

In [12]:
get_iocsANDexploits('CVE-2017-5638')

{'source': 'OTX alien vault / threatminer if alien vault empty',
 'Exploits': [{'author': 'Metasploit',
   'cve': 'CVE-2017-5638',
   'date': '1970-01-01',
   'description': 'struts_ognl',
   'name': "Apache Struts 2.3.5 < 2.3.31 / 2.5 < 2.5.10 - 'Jakarta' Multipart Parser OGNL Injection (Metasploit)",
   'platform': 'multiple',
   'port': '8080',
   'type': 'remote',
   'url': 'https://cwiki.apache.org/confluence/display/WW/S2-045'},
  {'author': 'CANVAS',
   'cve': 'CVE-2017-5638',
   'description': 'struts_ognl',
   'name': 'struts_ognl',
   'url': 'http://exploitlist.immunityinc.com/home/exploitpack/struts_ognl/struts_ognl'}],
 'pulse_context': {'count': 30,
  'samples': ['Bulbature, beneath the waves of GobRAT',
   'Gitpaste-12: Worming botnet spreading via GitHub and Pastebin',
   'Exposed Docker Server Abused to Drop Cryptominer DDoS Bot']},
 'iocs': {'IPs': [],
  'Domains': ['aduidc.xyz', 'beahh.com'],
  'URLs': ['http://info.beahh.com/e.png?id=okHfJRtbLzo&amp;amp;mac=3E-AA-40-